# **ML_PROJECT_GROUP**
#### This project focuses on developing a Machine Learning model to predict food prices across different regional markets in Tanzania. Food prices vary significantly between regions due to factors such as transportation costs, seasonal production, weather conditions, supply and demand, and local economic activities. These fluctuations can impact household food security, traders’ profitability, and government planning. By analyzing historical market data, including region, market location, commodity type, and time trends, the model learns patterns that influence price changes and generates accurate predictions. The system aims to support farmers, traders, policymakers, and consumers in making informed decisions through data-driven insights into food market trends.

In [ ]:
#import the required libraries
import numpy as np 
import pandas as pd 
import seaborn as sns 
import matplotlib.ticker as mtick  
import matplotlib.pyplot as plt
%matplotlib inline
import pandas as pd
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error



In [ ]:
food = pd.read_csv(r"C:\Users\Admin\Desktop\machine learning\food\Export.csv",on_bad_lines="skip")

In [ ]:
file_path = r"C:\Users\Admin\Desktop\machine learning\food\Export.csv"
with open(file_path, "r", encoding="utf-8") as f:
    total_rows = sum(1 for line in f)
print("Total rows in file:", total_rows)
print("Rows loaded in DataFrame:", loan.shape[0])
skipped_rows = total_rows - loan.shape[0] - 1  # minus 1 for header
print("Skipped rows:", skipped_rows)

To successfully load the dataset, the parameter on_bad_lines="skip" was used when reading the CSV file. This method was necessary because the original dataset contained some corrupted or improperly formatted rows that caused errors during loading. The file originally had 71,511 rows, but after loading it into pandas, only 62,485 rows were successfully read, meaning 9,025 problematic rows were skipped automatically. These skipped rows likely contained formatting issues such as extra commas or inconsistent column values. Using this method allowed the dataset to load without crashing the program and ensured that the remaining clean and valid data could still be used for analysis and machine learning model development.

In [ ]:
food.shape

In [ ]:
food.columns.values

### **Data preprocessing** 

In [ ]:
# Checking the data types of all the columns
food.dtypes

In [ ]:
#change the data type of date from object to datetime
food['date'] = pd.to_datetime(food['date'], errors='coerce')


In [ ]:
# Extract useful time features
food['year'] = food['date'].dt.year
food['month'] = food['date'].dt.month
food['day'] = food['date'].dt.day
food['week'] = food['date'].dt.isocalendar().week

In [ ]:
food.head()

In [ ]:
# Concise Summary of the dataframe, as we have too many columns, we are using the verbose = True mode
food.info(verbose = True) 

In [ ]:
#drop some columns that are not needed
food.drop(columns=['date', 'latitude', 'longitude', 'currency', 'usdprice'], inplace=True)

In [ ]:
# Concise Summary of the dataframe, as we have too many columns, we are using the verbose = True mode
food.info(verbose = True) 

### Dealing with missing value:

In [ ]:
missing = pd.DataFrame({
    'column': food.isnull().sum().index,
    'missing_pct': food.isnull().sum().values * 100 / loan1.shape[0]
})
plt.figure(figsize=(16,5))
ax = sns.pointplot(x='column', y='missing_pct', data=missing)
plt.xticks(rotation=90, fontsize=7)
plt.title("Percentage of Missing values")
plt.ylabel("PERCENTAGE")
plt.show()

### Missing Data - Initial Check
* All columns in the dataset (admin1, admin2, market, category, commodity, unit, priceflag, pricetype, price, year, month, day, week) have complete data with no missing values.<br>
* Notes on Handling Missing Data:<br>
* Columns with few missing values can be filled using mean/median (for numeric) or mode (for categorical).<br>
* Columns with many missing values (typically >30–40%) can be considered for removal if they add little analytical value.<br>
In this dataset, no imputation or column removal is required.

### Treatment of outliers:

In [ ]:
food.columns.values

In [ ]:
# boxplot all the numerical columns and see if there any outliers
for i in food[[ 'price', 'year', 'month', 'day', 'week']].columns:
    food.boxplot(column=i)
    plt.title(f"Box Plot Of {i}" , fontsize=20,
          color="orange")
    plt.show()

* Target Variable (price):<br>
price is the target variable in our prediction task. Outliers in the target reflect real extreme values in the market that the model should learn to predict.Removing or modifying these values could bias the model and reduce its ability to accurately predict high or low prices <br>
* Time-Related Features (year, month, day, week):<br>
These columns represent time information rather than continuous measurements.They have a fixed, meaningful range (e.g., month: 1–12, week: 1–52, day: 1–31), so any extreme values outside these ranges would indicate data errors, which are not present in this dataset.Since the values are valid and categorical in nature, treating them as outliers is not necessary.
* Conclusion: Outlier treatment is not required for any of these columns, as the target variable must retain its natural distribution and time-related features already have a valid fixed range.

In [ ]:
columns_to_skip = ['price', 'year', 'month', 'day', 'week']

for col in food.columns:
    if col not in columns_to_skip:
        print(f"Unique values in column '{col}':")
        print(food[col].nunique())
        print(food[col].unique())
        print("-"*50)  

In [ ]:
food.head()

### Features Encoding:

In [ ]:
# Identify categorical columns
object_columns = food.select_dtypes(include=['object','category']).columns

# Apply dummy encoding
food_dummies = pd.get_dummies(
    food,
    columns=object_columns,
    drop_first=True,   # avoid dummy variable trap
    dtype=int
)
food_dummies.head()

In [ ]:
#save the file of dummies 
food_dummies.to_csv("food_dummies.csv",index=False)

In [ ]:
food1 = food_dummies.copy()

### MODEL CREATION

In [ ]:
# Divide the data into dependents and independents
x=food1.drop(['price'], axis=1)
y=food1['price']

In [ ]:
x

In [ ]:
#variable for the x columns
X_columns = x.columns

In [ ]:
# After pd.get_dummies(x)
import pickle
with open("model_columns.pkl", "wb") as f:
    pickle.dump(x.columns, f)


In [ ]:
y

In [ ]:
# Splitting the data into training & test
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)
print(f"X_train:{len(x_train)}, X_test:{len(x_test)}")

In [ ]:
print(x_train.shape, y_train.shape)
print(x_test.shape,y_test.shape)

In [ ]:
# Feature Scaling
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test  = sc.transform(x_test)

In [ ]:
x_train

In [ ]:
x_test

In [ ]:
# Save the scaler
import pickle 

with open('scaler.sav', 'wb') as f:
    pickle.dump(sc, f)


# Decision Tree Regressor

In [ ]:
# defining the model
model_dt= DecisionTreeRegressor(max_depth=10,min_samples_leaf=5)
model_dt.fit(x_train,y_train)


In [ ]:
# Making Predictions
y_preddt=model_dt.predict(x_test)
y_preddt

#### Checking performance of the model genaral

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
mae = mean_absolute_error(y_test, y_preddt)
mse = mean_squared_error(y_test,y_preddt)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_preddt)
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R²:", r2)

#### Checking performmance of the model on Test and Train data

In [ ]:
# Making Predictions
y_predtr= model_dt.predict(x_train)
y_predte=model_dt.predict(x_test)


# Performance on train data
modeldt_tr_R2score=r2_score(y_train,y_predtr)
modeldt_tr_RMSE=np.sqrt(mean_squared_error(y_train,y_predtr))
modeldt_tr_MSE=mean_squared_error(y_train, y_predtr)
modeldt_tr_MAE=mean_absolute_error(y_train, y_predtr)

# Performance on test data
modeldt_te_R2score=r2_score(y_test,y_predte)
modeldt_te_RMSE=np.sqrt(mean_squared_error(y_test,y_predte))
modeldt_te_MSE=mean_squared_error(y_test,y_predte)
modeldt_te_MAE=mean_absolute_error(y_test,y_predte)

In [ ]:
# Print all metrics neatly
# -----------------------------
print("===== Decision Tree Performance =====\n")
print(">>> TRAIN SET METRICS <<<")
print(f"R² Score:           {modeldt_tr_R2score:.4f}")
print(f"RMSE:               {modeldt_tr_RMSE:.4f}")
print(f"MSE:                {modeldt_tr_MSE:.4f}")
print(f"MAE:                {modeldt_tr_MAE:.4f}")

print(">>> TEST SET METRICS <<<")
print(f"R² Score:           {modeldt_te_R2score:.4f}")
print(f"RMSE:               {modeldt_te_RMSE:.4f}")
print(f"MSE:                {modeldt_te_MSE:.4f}")
print(f"MAE:                {modeldt_te_MAE:.4f}")

* The Decision Tree model shows no evidence of overfitting, as the performance on the test set (R² = 0.9473) is slightly higher than the training set (R² = 0.9102). Additionally, the error metrics (RMSE and MAE) are comparable between training and testing data, indicating strong generalization ability.

### Visualization of the model

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt

plt.figure(figsize=(22,12))
plot_tree(
    model_dt,
    feature_names=x.columns,  
    class_names=["No Default", "Default"],
    filled=True,
    max_depth=3
)

plt.show()



# Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
# defining the model
lr = LinearRegression()
lr.fit(x_train,y_train)

In [ ]:
# Making Predictions
y_pred= lr.predict(x_test)
y_pred

#### Checking performance of the model genaral

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test,y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R²:", r2)

#### Checking performmance of the model on Test and Train data

In [ ]:
# Making Predictions
y_predtr= lr.predict(x_train)
y_predte=lr.predict(x_test)


# Performance on train data
modellr_tr_R2score=r2_score(y_train,y_predtr)
modellr_tr_RMSE=np.sqrt(mean_squared_error(y_train,y_predtr))
modellr_tr_MSE=mean_squared_error(y_train, y_predtr)
modellr_tr_MAE=mean_absolute_error(y_train, y_predtr)

# Performance on test data
modellr_te_R2score=r2_score(y_test,y_predte)
modellr_te_RMSE=np.sqrt(mean_squared_error(y_test,y_predte))
modellr_te_MSE=mean_squared_error(y_test,y_predte)
modellr_te_MAE=mean_absolute_error(y_test,y_predte)

In [ ]:
# Print all metrics neatly
# -----------------------------
print("===== Linear Regression Performance =====\n")
print(">>> TRAIN SET METRICS <<<")
print(f"R² Score:           {modellr_tr_R2score:.4f}")
print(f"RMSE:               {modellr_tr_RMSE:.4f}")
print(f"MSE:                {modellr_tr_MSE:.4f}")
print(f"MAE:                {modellr_tr_MAE:.4f}")

print(">>> TEST SET METRICS <<<")
print(f"R² Score:           {modellr_te_R2score:.4f}")
print(f"RMSE:               {modellr_te_RMSE:.4f}")
print(f"MSE:                {modellr_te_MSE:.4f}")
print(f"MAE:                {modellr_te_MAE:.4f}")

### Visualization of the model

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))
plt.scatter(y_test, y_predte)
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Actual vs Predicted (Linear Regression)")
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         color='red')  # Perfect prediction line
plt.show()


* The Multiple Linear Regression model shows consistent and reliable predictive performance across both training and testing datasets. There is no evidence of overfitting, and the model explains a substantial proportion of the variation in food prices. However, further improvements could be achieved through feature engineering, model refinement, or the use of more flexible modeling approaches.

* **Conclusion:** <br>
Based on the evaluation metrics, the Decision Tree regression model significantly outperforms the Multiple Linear Regression model in predicting food prices. It achieves higher explanatory power (R²) and substantially lower prediction errors (RMSE and MAE).Therefore, the Decision Tree model is the more suitable model for this dataset, as it better captures the complex relationships present in the food price data.

In [ ]:
#save the model
import pickle
with open('finalized_model.sav', 'wb') as file:
    pickle.dump(model_dt, file)


In [ ]:
with open('finalized_model.sav', 'rb') as file:
    loaded_model = pickle.load(file)

# Make predictions
y_pred_final = loaded_model.predict(x_test)

In [ ]:
y_pred_final[1]

In [ ]:
print("Enter Market Price Prediction Features:")

# Categorical inputs
admin1 = input("Admin1 (e.g., Kabul): ")
admin2 = input("Admin2 (e.g., Kabul City): ")
market = input("Market name: ")
category = input("Category (e.g., cereals): ")
commodity = input("Commodity (e.g., Wheat): ")
unit = input("Unit (e.g., kg): ")
priceflag = input("Price flag (e.g., official): ")
pricetype = input("Price type (e.g., retail): ")

# Numerical inputs
year = int(input("Year (e.g., 2024): "))
month = int(input("Month (1-12): "))
day = int(input("Day (1-31): "))
week = int(input("Week number (1-52): "))

# Create DataFrame (same structure as training before encoding)
input_dict = {
    'admin1': admin1,
    'admin2': admin2,
    'market': market,
    'category': category,
    'commodity': commodity,
    'unit': unit,
    'priceflag': priceflag,
    'pricetype': pricetype,
    'year': year,
    'month': month,
    'day': day,
    'week': week
}

input_df = pd.DataFrame([input_dict])

# Apply get_dummies (same as training)
input_encoded = pd.get_dummies(input_df)

# Align columns with training data
input_encoded = input_encoded.reindex(columns=X_columns, fill_value=0)

# Apply feature scaling
input_scaled = sc.transform(input_encoded)

# Predict price
predicted_price = loaded_model.predict(input_scaled)

print("\n📊 Predicted Market Price:", predicted_price[0])


In [ ]:
Arusha	Arusha Urban	Arusha (urban)	cereals and tubers	Maize	100 KG	actual	Wholesale	38464.29	2006	1	15	2